<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/07_robust_modeling_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Robust Modeling Workflow

In this lesson we extend the six-step workflow to introduce critical modeling best practices. In this session, we will move from classfication to regression.We will reuse the King County Housing dataset introduced in the last module for this purpose.

In [25]:
# tools from standard library
import functools
from pathlib import Path

# tools for data wrangling
import numpy as np
import polars as pl

import geopandas as gpd

# pipeline and transformers
from sklearn.pipeline import (
    Pipeline,
    make_pipeline,
    )
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer,
    )
from sklearn.impute import SimpleImputer

# models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor

# metrics for evaluation
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics

# cross validation and dataset splitter
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    cross_validate,
    KFold
    )

# base types for custom transformer
from sklearn.base import BaseEstimator, TransformerMixin

# sklearn config to outpur polars dataframes
from sklearn import set_config
set_config(transform_output='polars')


In [3]:
data_root = Path("/content/drive/MyDrive/dataprogpy/data")
house_data_path = Path("kc_house_data.csv")
schdst_file_path = Path("School_Districts_in_King_County___schdst_area/School_Districts_in_King_County___schdst_area.shp")

In [4]:
kc_schdst = gpd.read_file(data_root / schdst_file_path) # school district shapes

In [5]:
# Load the dataset
raw = pl.read_csv(data_root / house_data_path)
print(kc_schdst.columns)
raw.head()

Index(['OBJECTID', 'SCHDST', 'NAME', 'DSTNUM', 'Shape_Leng', 'Shape_Area',
       'geometry'],
      dtype='object')


id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
i64,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64
7129300520,"""20141013T000000""",221900.0,3,1.0,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
6414100192,"""20141209T000000""",538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.721,-122.319,1690,7639
5631500400,"""20150225T000000""",180000.0,2,1.0,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
2487200875,"""20141209T000000""",604000.0,4,3.0,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
1954400510,"""20150218T000000""",510000.0,3,2.0,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [6]:
raw_pd = raw.to_pandas()
raw_pd = gpd.GeoDataFrame(
        raw_pd,
        geometry=gpd.points_from_xy(raw_pd.long, raw_pd.lat),
        crs="EPSG:4326"
        ).sjoin(
            kc_schdst.to_crs("EPSG:4326"),
            how="left",
            predicate="intersects")
print(raw_pd.shape)
# raw_pd.head()
raw = pl.from_pandas(raw_pd[raw.columns + ["NAME"]]).select(
    pl.col(raw.columns),
    pl.col("NAME").alias("school_district")
)
print(raw.shape)
print(raw.columns)
raw.head()

(21613, 29)
(21613, 22)
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'school_district']


id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,school_district
i64,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,str
7129300520,"""20141013T000000""",221900.0,3,1.0,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650,"""Seattle"""
6414100192,"""20141209T000000""",538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.721,-122.319,1690,7639,"""Seattle"""
5631500400,"""20150225T000000""",180000.0,2,1.0,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062,"""Northshore"""
2487200875,"""20141209T000000""",604000.0,4,3.0,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000,"""Seattle"""
1954400510,"""20150218T000000""",510000.0,3,2.0,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503,"""Lake Washington"""


In [7]:
def tweak_housing(df):
    return (
        df.with_columns(
            zipcode=pl.col('zipcode').cast(pl.String).cast(pl.Categorical),
            school_district=pl.col('school_district').cast(pl.Categorical),
            date=pl.col('date').str.strptime(pl.Date, format="%Y%m%dT%H%M%S"),
            yr_renovated=pl.col('yr_renovated').replace(0, None),
            ).select(
                pl.col(['id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'school_district','lat', 'long', 'sqft_living15', 'sqft_lot15', 'date', ])
            )
    )

tweak_transformer = FunctionTransformer(tweak_housing)

In [8]:
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
                    'floors', 'waterfront', 'view', 'condition', 'grade',
                    'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated',
                    'lat', 'long', 'sqft_living15', 'sqft_lot15', ]

numeric_transformer = Pipeline(
    steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ])

In [9]:
categorical_features = ['zipcode', 'school_district']

categorical_transformer = Pipeline(
    steps=[
         ('onehot', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False)),
    # ('target', TargetEncoder()),
    # ('std', StandardScaler()),
    ])

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat',categorical_transformer, categorical_features),
        ],
        # remainder='passthrough',
    )

In [17]:
lr =  LinearRegression()
X = raw
y = raw.select('price')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

lr_pipe = Pipeline(steps=[
    ('tweak', tweak_transformer),
    ('preprocessor', preprocessor),
    ('lr', lr),
    ])

lr_pipe.fit(X_train, y_train)
print(lr_pipe.score(X_test, y_test))
lr_pipe

0.8010475068148948


Pipeline(steps=[('tweak',
                 FunctionTransformer(func=<function tweak_housing at 0x780cd7b902c0>)),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedrooms', 'bathrooms',
                                                   'sqft_living', 'sqft_lot',
                                                   'floors', 'waterfront',
                                                   'view', 'condition', 'grade',
                                                   'sqft_above',
                                                   'sqft_basement', 'yr_built',
                                                   'yr_renovated', 'lat',
                                                   'long', 'sqft_living15',
                                                   'sqft_lot15']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['zipcode',
                                                   'school_district'])])),
                ('lr', LinearRegression())])

In [21]:
def train_test(model, test_size=0.4, random_state=42):
  X = raw
  y = raw.select('price')

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

  pipeline = make_pipeline(tweak_transformer, preprocessor, model)
  pipeline.fit(X_train, y_train)
  print(pipeline.score(X_test, y_test))
  return pipeline

In [26]:
train_test(DummyRegressor(strategy='median'), 0.4,)

-0.0597493285675994


Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function tweak_housing at 0x780cd7b902c0>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedrooms', 'bathrooms',
                                                   'sqft_living', 'sqft_lot',
                                                   'floors', 'waterfront',
                                                   'view', 'condition', 'grade',
                                                   'sqft_above',
                                                   'sqft_basement', 'yr_built',
                                                   'yr_renovated', 'lat',
                                                   'long', 'sqft_living15',
                                                   'sqft_lot15']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['zipcode',
                                                   'school_district'])])),
                ('dummyregressor', DummyRegressor(strategy='median'))])

In [22]:
train_test(lr, 0.4,)

0.8010475068148948


Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function tweak_housing at 0x780cd7b902c0>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedrooms', 'bathrooms',
                                                   'sqft_living', 'sqft_lot',
                                                   'floors', 'waterfront',
                                                   'view', 'condition', 'grade',
                                                   'sqft_above',
                                                   'sqft_basement', 'yr_built',
                                                   'yr_renovated', 'lat',
                                                   'long', 'sqft_living15',
                                                   'sqft_lot15']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['zipcode',
                                                   'school_district'])])),
                ('linearregression', LinearRegression())])

In [24]:
train_test(DecisionTreeRegressor(max_depth=5, random_state=42), 0.6,)

0.6970429193462337


Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function tweak_housing at 0x780cd7b902c0>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedrooms', 'bathrooms',
                                                   'sqft_living', 'sqft_lot',
                                                   'floors', 'waterfront',
                                                   'view', 'condition', 'grade',
                                                   'sqft_above',
                                                   'sqft_basement', 'yr_built',
                                                   'yr_renovated', 'lat',
                                                   'long', 'sqft_living15',
                                                   'sqft_lot15']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['zipcode',
                                                   'school_district'])])),
                ('decisiontreeregressor',
                 DecisionTreeRegressor(max_depth=5, random_state=42))])

## References

1. [Comparing cross-validation strategies](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html)
1.

In [13]:
dir(lr)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_tags__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_decision_function',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_set_intercept',
 '_validate_data',
 '_validate_params',
 'coef_',
 'copy_X',
 'featur